# PicoRV32 + CIM SoC — 环境准备与验证本 notebook 在**宿主机** (x86 Linux) 上运行，完成以下工作：1. 检查工具链 (RISC-V GCC, Python)2. 训练 Small MLP (784→16→10) 并量化3. Python golden model 验证 20 张测试图4. 编译固件并检查 BRAM 占用5. 打包 PYNQ 部署文件完成后可以：- 运行 `bash hw/scripts/run_tb_rv32_batch.sh` 进行 VCS 批量仿真- 将打包文件上传到 PYNQ 板进行上板验证

## 0. 路径设置

In [ ]:
import os, sys, subprocess, shutil
import numpy as np

# 项目根目录 (picorv32/)
PROJ_ROOT = os.path.abspath(".")
FW_DIR    = os.path.join(PROJ_ROOT, "fw")
HW_DIR    = os.path.join(PROJ_ROOT, "hw")
DATA_DIR  = "small_mlp_data"
N_IMAGES  = 20

assert os.path.exists(os.path.join(FW_DIR, "firmware.c")),     f"请在 picorv32/ 目录下运行本 notebook (当前: {PROJ_ROOT})"
print(f"Project root: {PROJ_ROOT}")
print(f"Firmware dir: {FW_DIR}")

## 1. 检查工具链

In [ ]:
# RISC-V GCC
rv_gcc = None
for prefix in ["riscv64-elf-", "riscv64-unknown-elf-", "riscv32-unknown-elf-"]:
    if shutil.which(f"{prefix}gcc"):
        rv_gcc = f"{prefix}gcc"
        break

if rv_gcc:
    ver = subprocess.check_output([rv_gcc, "--version"], text=True).split("\n")[0]
    print(f"✓ RISC-V GCC: {ver}")
else:
    print("✗ RISC-V GCC 未找到!")
    print("  安装: sudo pacman -S riscv64-elf-gcc  (Arch)")
    print("     或: sudo apt install gcc-riscv64-unknown-elf  (Debian/Ubuntu)")

# Python packages
try:
    import torch
    print(f"✓ PyTorch {torch.__version__}")
except ImportError:
    print("✗ PyTorch 未安装: pip install torch torchvision")

try:
    from torchvision import datasets
    print("✓ torchvision")
except ImportError:
    print("✗ torchvision 未安装: pip install torchvision")

## 2. 训练 Small MLP 并量化

In [ ]:
os.chdir(FW_DIR)

# 训练 + 量化 + 导出 (784→16→10, fits in 32KB BRAM)
if not os.path.isdir(DATA_DIR):
    print(f"训练模型并导出到 {DATA_DIR}/ ...")
    ret = subprocess.run(
        ["python3", "small_mlp_quantize.py",
         "--output-dir", DATA_DIR,
         "--num-test", str(N_IMAGES),
         "--seed", "42"],
        capture_output=False, text=True
    )
    assert ret.returncode == 0, "训练失败!"
else:
    print(f"{DATA_DIR}/ 已存在, 跳过训练")

# 列出生成的文件
for f in sorted(os.listdir(DATA_DIR)):
    if not f.startswith('.'):
        full = os.path.join(DATA_DIR, f)
        if os.path.isfile(full):
            print(f"  {f:30s} {os.path.getsize(full):>8d} B")
        else:
            n = len(os.listdir(full))
            print(f"  {f + '/':30s} ({n} files)")

## 3. Python Golden Model 验证 (20 张图)

In [ ]:
from torchvision import datasets, transforms
import struct

def read_hex_u32(path):
    with open(path) as f:
        return [int(l.strip(), 16) for l in f if l.strip()]

def read_hex_u8(path):
    with open(path) as f:
        return [int(l.strip(), 16) & 0xFF for l in f if l.strip()]

# 加载量化参数
qp = read_hex_u32(f"{DATA_DIR}/quant_params.hex")
zps = read_hex_u32(f"{DATA_DIR}/zero_points.hex")
fc1_mult, fc1_shift = qp[0], qp[1]
fc2_mult, fc2_shift = qp[2], qp[3]
hw_zp1 = np.int32(np.uint32(zps[0]))
hw_zp2 = np.int32(np.uint32(zps[1]))

# 从 hex 文件解包权重
def unpack_weights_from_chunks(chunk_path, out_dim, in_dim):
    chunks = read_hex_u32(chunk_path)
    TILE_R, TILE_C = 16, 16
    n_ob = (out_dim + TILE_R - 1) // TILE_R
    n_ib = (in_dim + TILE_C - 1) // TILE_C
    w = np.zeros((out_dim, in_dim), dtype=np.int8)
    idx = 0
    for ob in range(n_ob):
        for ib in range(n_ib):
            for ch in range(64):  # CHUNKS_PER_TILE
                r = ch // 4
                cg = ch % 4
                word = chunks[idx]; idx += 1
                for b in range(4):
                    oi = ob * TILE_R + r
                    ii = ib * TILE_C + cg * 4 + b
                    if oi < out_dim and ii < in_dim:
                        val = (word >> (b*8)) & 0xFF
                        w[oi, ii] = np.int8(struct.unpack('b', bytes([val]))[0])
    return w

def unpack_bias(path):
    vals = read_hex_u32(path)
    return np.array([np.int32(np.uint32(v)) for v in vals], dtype=np.int32)

# 硬件精确 MVM
def hw_mvm(x_u8, w_i8, b_i32, zp, mult, shift, relu):
    x_eff = np.clip(x_u8.astype(np.int32) - zp, 0, 511)
    acc = w_i8.astype(np.int32) @ x_eff + b_i32
    if relu:
        acc = np.maximum(acc, 0)
    out = np.zeros(len(acc), dtype=np.int8)
    for i in range(len(acc)):
        prod = int(acc[i]) * int(mult)
        shifted = (prod + (1 << (shift-1))) >> shift if shift > 0 else prod
        out[i] = np.int8(max(-128, min(127, shifted)))
    return out

# 加载模型
w1 = unpack_weights_from_chunks(f"{DATA_DIR}/fc1_weight_tiles.hex", 16, 784)
b1 = unpack_bias(f"{DATA_DIR}/fc1_bias.hex")
w2 = unpack_weights_from_chunks(f"{DATA_DIR}/fc2_weight_tiles.hex", 10, 16)
b2 = unpack_bias(f"{DATA_DIR}/fc2_bias.hex")

print(f"FC1: w={w1.shape}, b={b1.shape}, mult={fc1_mult}, shift={fc1_shift}, zp={hw_zp1}")
print(f"FC2: w={w2.shape}, b={b2.shape}, mult={fc2_mult}, shift={fc2_shift}, zp={hw_zp2}")

# 推理 20 张图
results = []
for i in range(N_IMAGES):
    prefix = f"img_{i:04d}"
    img = np.array(read_hex_u8(f"{DATA_DIR}/test_images/{prefix}.hex"), dtype=np.uint8)
    label = int(open(f"{DATA_DIR}/test_images/{prefix}_label.txt").read().strip())

    fc1_out = hw_mvm(img, w1, b1, hw_zp1, fc1_mult, fc1_shift, True)
    fc2_in = fc1_out.view(np.uint8)
    fc2_out = hw_mvm(fc2_in, w2, b2, hw_zp2, fc2_mult, fc2_shift, False)
    pred = int(np.argmax(fc2_out))

    ok = "✓" if pred == label else "✗"
    results.append((i, label, pred, pred == label))
    print(f"  [{i:04d}] label={label}, pred={pred} {ok}")

correct = sum(1 for r in results if r[3])
print(f"\nGolden model accuracy: {correct}/{N_IMAGES} ({100*correct/N_IMAGES:.0f}%)")

## 4. 编译固件并检查 BRAM 占用

In [ ]:
os.chdir(FW_DIR)

# 确定 CROSS 前缀
cross_prefix = rv_gcc.replace("gcc", "") if rv_gcc else "riscv64-elf-"

# 编译 image 0 作为示例
subprocess.run(["make", "-s", "clean"], check=False)
ret = subprocess.run(
    ["make", f"DATA_DIR={DATA_DIR}", "IMAGE_IDX=0", f"CROSS={cross_prefix}"],
    capture_output=True, text=True
)
print(ret.stdout)
if ret.returncode != 0:
    print("编译失败:")
    print(ret.stderr)
else:
    # 读取 size 输出
    size_out = subprocess.check_output(
        [f"{cross_prefix}size", "firmware.elf"], text=True)
    print(size_out)

    # 检查 firmware.hex
    n_words = sum(1 for l in open("firmware.hex") if l.strip() and not l.startswith('@'))
    n_bytes = n_words * 4
    print(f"firmware.hex: {n_words} words = {n_bytes} bytes ({n_bytes/1024:.1f} KB)")
    print(f"BRAM 容量: 32768 bytes (32 KB)")
    print(f"占用率: {100*n_bytes/32768:.1f}%")

    if n_bytes > 32768:
        print("⚠ 超出 32KB BRAM! 请使用 small_mlp_data")
    else:
        print("✓ 固件可以放入 BRAM")

## 5. 打包 PYNQ 部署文件

In [ ]:
os.chdir(FW_DIR)

# 将 PYNQ 需要的文件打包到一个目录
pynq_pkg_dir = os.path.join(PROJ_ROOT, "pynq_deploy")
os.makedirs(pynq_pkg_dir, exist_ok=True)

# 复制模型数据
import shutil as sh
deploy_data = os.path.join(pynq_pkg_dir, DATA_DIR)
if os.path.exists(deploy_data):
    sh.rmtree(deploy_data)
sh.copytree(os.path.join(FW_DIR, DATA_DIR), deploy_data)

# 复制 golden model 结果
with open(os.path.join(pynq_pkg_dir, "golden_results.txt"), "w") as f:
    f.write("# image_idx label pred correct\n")
    for idx, label, pred, ok in results:
        f.write(f"{idx} {label} {pred} {1 if ok else 0}\n")

# 复制 PYNQ notebook
pynq_nb_src = os.path.join(PROJ_ROOT, "picorv32_small_mlp_pynq.ipynb")
if os.path.exists(pynq_nb_src):
    sh.copy2(pynq_nb_src, pynq_pkg_dir)

print(f"打包完成: {pynq_pkg_dir}/")
for f in sorted(os.listdir(pynq_pkg_dir)):
    print(f"  {f}")
print()
print("上传以下文件到 PYNQ:")
print("  1. bitstream&hwh/checkpoint1/cim_soc.bit + cim_soc.hwh")
print(f"  2. pynq_deploy/{DATA_DIR}/ (整个目录)")
print("  3. pynq_deploy/golden_results.txt")
print("  4. picorv32_small_mlp_pynq.ipynb")
print("  5. sw/cim_driver.py")

## 下一步- **VCS 批量仿真**: `cd picorv32 && bash hw/scripts/run_tb_rv32_batch.sh`- **PYNQ 上板**: 上传文件后运行 `picorv32_small_mlp_pynq.ipynb`- 两边结果应该 **bit-exact 一致**（同模型、同量化、同硬件）